In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

DATASET_DIR = "../DATASET/dataset_stage3_residual"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")
TEST_DIR = os.path.join(DATASET_DIR, "test")

# Testing on real-life images (not used for training/validation)
REAL_LIFE_DIR = "../DATASET/dataset_stage3_residual/real_world"

# Training schedule
EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 20
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5

# Fine-tuning
N_UNFROZEN = 40  # unfreeze last N layers (BN stays frozen)

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
def _random_jpeg_quality(x, min_q=40, max_q=95):
    return x


def _gaussian_kernel(ksize: int, sigma: float):
    ax = tf.range(-ksize // 2 + 1, ksize // 2 + 1, dtype=tf.float32)
    xx, yy = tf.meshgrid(ax, ax)
    kernel = tf.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / tf.reduce_sum(kernel)
    kernel = kernel[:, :, tf.newaxis, tf.newaxis]
    return kernel


def _gaussian_blur(x, ksize=5, sigma_min=0.3, sigma_max=1.5):
    sigma = tf.random.uniform([], sigma_min, sigma_max)
    kernel = _gaussian_kernel(ksize, sigma)
    x4 = x[tf.newaxis, ...]
    kernel = tf.tile(kernel, [1, 1, tf.shape(x)[-1], 1])  # (k,k,C,1)
    x_blur = tf.nn.depthwise_conv2d(x4, kernel, strides=[1, 1, 1, 1], padding="SAME")
    return x_blur[0]


def _add_sensor_noise(x, std_min=0.0, std_max=12.0):
    std = tf.random.uniform([], std_min, std_max)
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=std, dtype=tf.float32)
    return tf.clip_by_value(x + noise, 0.0, 255.0)


def _random_crop_resize(x, y, min_scale=0.6, max_scale=1.0):
    h = tf.shape(x)[0]
    w = tf.shape(x)[1]
    scale = tf.random.uniform([], min_scale, max_scale)
    new_h = tf.maximum(tf.cast(tf.cast(h, tf.float32) * scale, tf.int32), 1)
    new_w = tf.maximum(tf.cast(tf.cast(w, tf.float32) * scale, tf.int32), 1)
    crop_size = tf.stack([new_h, new_w, tf.shape(x)[-1]])
    x = tf.image.random_crop(x, size=crop_size, seed=SEED)
    x = tf.image.resize(x, IMG_SIZE, method="bilinear")
    return x, y


random_rotation = tf.keras.layers.RandomRotation(factor=20.0 / 360.0, fill_mode="reflect")


def _augment_single_example(x, y):
    x, y = _random_crop_resize(x, y)

    x = tf.image.random_flip_left_right(x, seed=SEED)

    do_rot90 = tf.random.uniform([]) < 0.20
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x = tf.cond(do_rot90, lambda: tf.image.rot90(x, k=k), lambda: x)

    do_tilt = tf.random.uniform([]) < 0.60
    x = tf.cond(
        do_tilt,
        lambda: random_rotation(x[tf.newaxis, ...], training=True)[0],
        lambda: x,
    )

    x = tf.image.random_brightness(x, max_delta=0.25)
    x = tf.image.random_contrast(x, lower=0.6, upper=1.4)
    x = tf.image.random_saturation(x, lower=0.7, upper=1.3)
    x = tf.image.random_hue(x, max_delta=0.05)

    x = tf.cond(tf.random.uniform([]) < 0.35, lambda: _gaussian_blur(x, ksize=5), lambda: x)
    x = tf.cond(tf.random.uniform([]) < 0.50, lambda: _add_sensor_noise(x), lambda: x)

    x = preprocess_input(x)
    return x, y


def camera_like_augment(x, y):
    # image_dataset_from_directory returns batched tensors, so augment each image in the batch.
    if x.shape.rank == 4:
        x, y = tf.map_fn(
            lambda sample: _augment_single_example(sample[0], sample[1]),
            (x, y),
            fn_output_signature=(tf.float32, tf.float32),
        )
        return x, y

    return _augment_single_example(x, y)


def basic_preprocess(x, y):
    return preprocess_input(x), y

In [ ]:
def camera_like_augment(x, y):
    # x is float32 in [0,255] from image_dataset_from_directory
    x, y = _random_crop_resize(x, y)

    # realistic flip
    x = tf.image.random_flip_left_right(x, seed=SEED)

    # 90° rotation sometimes (orientation)
    do_rot90 = tf.random.uniform([]) < 0.20
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x = tf.cond(do_rot90, lambda: tf.image.rot90(x, k=k), lambda: x)

    # tilt: small rotation
    do_tilt = tf.random.uniform([]) < 0.60
    angle_deg = tf.random.uniform([], -20.0, 20.0)
    angle_rad = angle_deg * (np.pi / 180.0)
    x = tf.cond(
        do_tilt,
        lambda: tf.image.rotate(x, angles=angle_rad, fill_mode="reflect"),
        lambda: x
    )

    # lighting/color
    x = tf.image.random_brightness(x, max_delta=0.25)
    x = tf.image.random_contrast(x, lower=0.6, upper=1.4)
    x = tf.image.random_saturation(x, lower=0.7, upper=1.3)
    x = tf.image.random_hue(x, max_delta=0.05)

    # blur/noise/jpeg sometimes
    x = tf.cond(tf.random.uniform([]) < 0.35, lambda: _gaussian_blur(x, ksize=5), lambda: x)
    x = tf.cond(tf.random.uniform([]) < 0.50, lambda: _add_sensor_noise(x), lambda: x)
    x = tf.cond(tf.random.uniform([]) < 0.35, lambda: _random_jpeg_quality(x), lambda: x)

    # mobilenet preprocessing (keep consistent with backend)
    x = preprocess_input(x)
    return x, y

def basic_preprocess(x, y):
    return preprocess_input(x), y

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Class names:", class_names)
print("NUM_CLASSES:", NUM_CLASSES)

real_life_ds = None
if REAL_LIFE_DIR is not None and os.path.isdir(REAL_LIFE_DIR):
    real_life_ds = image_dataset_from_directory(
        REAL_LIFE_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False
    )
    print("Real-life class names:", real_life_ds.class_names)
else:
    print("Real-life class names: N/A (REAL_LIFE_DIR not found)")

train_ds = train_ds.map(camera_like_augment, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

if real_life_ds is not None:
    real_life_ds = real_life_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
base_model = MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()

In [ ]:
def compile_model(lr: float):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_acc"),
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),

            # One-vs-rest PR-AUC over classes (macro-ish behavior)
            tf.keras.metrics.AUC(name="pr_auc_ovr", curve="PR", multi_label=True, num_labels=NUM_CLASSES),
        ],
    )

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

class MacroPRCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, num_classes, name_prefix=""):
        super().__init__()
        self.val_ds = val_ds
        self.num_classes = num_classes
        self.name_prefix = name_prefix

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        y_true = []
        y_prob = []
        for xb, yb in self.val_ds:
            p = self.model.predict(xb, verbose=0)
            y_prob.append(p)
            y_true.append(np.argmax(yb.numpy(), axis=1))

        y_true = np.concatenate(y_true)
        y_prob = np.concatenate(y_prob, axis=0)
        y_pred = np.argmax(y_prob, axis=1)

        macro_prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
        macro_rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

        logs["precision"] = macro_prec
        logs["recall"] = macro_rec
        # Keras will automatically prefix val_? No—so we store them as val_precision/val_recall:
        logs["val_precision"] = macro_prec
        logs["val_recall"] = macro_rec

        print(f"\n[macro] val_precision={macro_prec:.4f} val_recall={macro_rec:.4f}")

macro_cb = MacroPRCallback(val_ds=val_ds, num_classes=NUM_CLASSES)

In [ ]:
base_model.trainable = False
compile_model(LR_PHASE1)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc_ovr", mode="max", patience=3, restore_best_weights=True),
    macro_cb,
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

for layer in base_model.layers[:-N_UNFROZEN]:
    layer.trainable = False

compile_model(LR_PHASE2)

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc_ovr", mode="max", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc_ovr", mode="max", factor=0.5, patience=2, min_lr=1e-7),
    macro_cb,
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2
)

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    keys = set(h1.history.keys()) | set(h2.history.keys())
    for k in keys:
        merged[k] = list(h1.history.get(k, [])) + list(h2.history.get(k, []))
    return merged

merged = merge_histories(history1, history2)

plt.figure(figsize=(12, 7))

plt.subplot(2, 2, 1)
plt.plot(merged.get("accuracy", []), label="Train Accuracy")
plt.plot(merged.get("val_accuracy", []), label="Val Accuracy")
plt.legend(); plt.title("Accuracy")

plt.subplot(2, 2, 2)
plt.plot(merged.get("loss", []), label="Train Loss")
plt.plot(merged.get("val_loss", []), label="Val Loss")
plt.legend(); plt.title("Loss")

plt.subplot(2, 2, 3)
plt.plot(merged.get("precision", []), label="Train Precision (macro)")
plt.plot(merged.get("val_precision", []), label="Val Precision (macro)")
plt.legend(); plt.title("Precision (macro)")

plt.subplot(2, 2, 4)
plt.plot(merged.get("recall", []), label="Train Recall (macro)")
plt.plot(merged.get("val_recall", []), label="Val Recall (macro)")
plt.legend(); plt.title("Recall (macro)")

plt.tight_layout()
plt.show()

In [ ]:
def report_and_cm_multiclass(ds, class_names, title=""):
    y_true = []
    y_probs = []

    for xb, yb in ds:
        probs = model.predict(xb, verbose=0)
        y_probs.append(probs)
        y_true.append(np.argmax(yb.numpy(), axis=1))

    y_probs = np.concatenate(y_probs, axis=0)
    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.argmax(y_probs, axis=1)

    print("\n" + title)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix - {title}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

# TEST confusion matrix
report_and_cm_multiclass(test_ds, class_names, title="TEST")

# REAL-LIFE confusion matrix (if you have it)
if real_life_ds is not None:
    report_and_cm_multiclass(real_life_ds, class_names, title="REAL-LIFE")

In [ ]:
print("\nDataset test evaluation:")
model.evaluate(test_ds, verbose=2)

if real_life_ds is not None:
    print("\nReal-life evaluation:")
    model.evaluate(real_life_ds, verbose=2)

In [ ]:
def collect_probs_and_labels(ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0)  # (B, C)
        y_prob.append(p)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true, axis=0)  # one-hot
    y_prob = np.concatenate(y_prob, axis=0)
    y_true_idx = np.argmax(y_true, axis=1)
    return y_true, y_true_idx, y_prob

cal_ds = real_life_ds if real_life_ds is not None else test_ds  # fallback, but real-life is strongly preferred
y_true_oh, y_true_idx, y_prob = collect_probs_and_labels(cal_ds)

print("Calibration dataset size:", len(y_true_idx))

In [ ]:
def calibrate_uncertainty_thresholds(
    y_true_idx,
    y_prob,
    num_classes,
    t_grid=np.linspace(0.40, 0.90, 11),
    margin_grid=np.linspace(0.00, 0.30, 7),
    min_recall_per_class=0.70,   # increase as you collect more real-life images
):
    # Per-class threshold uses same t for all classes initially (simple, stable)
    # You can extend to per-class thresholds later.
    best = None

    p_sorted = np.sort(y_prob, axis=1)
    p1 = p_sorted[:, -1]
    p2 = p_sorted[:, -2]
    pred = np.argmax(y_prob, axis=1)

    for t in t_grid:
        for m in margin_grid:
            accept = (p1 >= t) & ((p1 - p2) >= m)
            coverage = float(np.mean(accept))

            # compute per-class recall among samples of that true class:
            ok = True
            recalls = []
            for c in range(num_classes):
                mask = (y_true_idx == c)
                if not np.any(mask):
                    # no samples of this class in calibration set -> can't constrain it
                    recalls.append(np.nan)
                    continue
                recall_c = float(np.mean((pred[mask] == c) & accept[mask]))
                recalls.append(recall_c)
                if recall_c < min_recall_per_class:
                    ok = False
                    break
            if not ok:
                continue

            candidate = {"t": float(t), "margin": float(m), "coverage": coverage, "recalls": recalls}
            if best is None or candidate["coverage"] > best["coverage"]:
                best = candidate

    return best

best_u = calibrate_uncertainty_thresholds(
    y_true_idx=y_true_idx,
    y_prob=y_prob,
    num_classes=NUM_CLASSES,
    min_recall_per_class=0.70
)

if best_u is None:
    print("No thresholds met constraints. Relax min_recall_per_class (e.g., 0.6) or collect more real-life images.")
else:
    print("Chosen uncertainty params:")
    print("  accept if max_prob >= t AND (p1 - p2) >= margin")
    print(f"  t = {best_u['t']:.2f}")
    print(f"  margin = {best_u['margin']:.2f}")
    print(f"  coverage = {best_u['coverage']:.3f}")
    for i, r in enumerate(best_u["recalls"]):
        print(f"  recall[{class_names[i]}] = {r}")

In [ ]:
def apply_uncertainty_policy(y_prob, t, margin):
    p_sorted = np.sort(y_prob, axis=1)
    p1 = p_sorted[:, -1]
    p2 = p_sorted[:, -2]
    pred = np.argmax(y_prob, axis=1)
    accept = (p1 >= t) & ((p1 - p2) >= margin)
    pred_or_uncertain = pred.copy()
    pred_or_uncertain[~accept] = -1
    return pred_or_uncertain, accept

if best_u is not None:
    pred_u, accept = apply_uncertainty_policy(y_prob, best_u["t"], best_u["margin"])
    print("Uncertain rate:", float(np.mean(~accept)))

    # confusion matrix on accepted only
    accepted_idx = np.where(accept)[0]
    if len(accepted_idx) > 0:
        y_true_acc = y_true_idx[accepted_idx]
        y_pred_acc = pred_u[accepted_idx]

        cm = tf.math.confusion_matrix(y_true_acc, y_pred_acc, num_classes=NUM_CLASSES).numpy()
        print("Confusion matrix (accepted samples only):")
        print(cm)
    else:
        print("No accepted samples under current thresholds.")

In [ ]:
OUT_DIR = "./enhanced-model"
os.makedirs(OUT_DIR, exist_ok=True)

tflite_path = os.path.join(OUT_DIR, "stage3_residual_model.tflite")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("Saved TFLite model to:", tflite_path)

if best_u is not None:
    np.savez(
        os.path.join(OUT_DIR, "residual_uncertainty_params.npz"),
        t=best_u["t"],
        margin=best_u["margin"],
        class_names=np.array(class_names, dtype=object),
    )
    print("Saved uncertainty params.")